In [1]:
import pandas as pd
import numpy as np
import json
import re

In [2]:
listing_data = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\data\raw\listings.csv.gz")

In [3]:
print(listing_data.head().to_string())

     id                        listing_url       scrape_id last_scraped           source                                               name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         description  neighborhood_overview                                                                                               picture_url  host_id                                 host_url  host_profile_id                                          host_profile_url host_name  host_since  hosts_time_as_user_years  hosts_time_as_user_months  hosts_time_as_

In [4]:
listing_data.shape

(35036, 90)

In [5]:
listing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 35036 entries, 0 to 35035
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            35036 non-null  int64  
 1   listing_url                                   35036 non-null  str    
 2   scrape_id                                     35036 non-null  int64  
 3   last_scraped                                  35036 non-null  str    
 4   source                                        35036 non-null  str    
 5   name                                          35034 non-null  str    
 6   description                                   33517 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   34901 non-null  str    
 9   host_id                                       35036 non-null  int64  
 1

In [6]:
listing_data.describe()

,id,scrape_id,neighborhood_overview,host_id,host_profile_id,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,3.503600e+04,3.503600e+04,0.0,3.503600e+04,3.419600e+04,0.0,34196.000000,34196.000000,34196.000000,34196.000000,...,24528.000000,24536.000000,24525.000000,24526.00000,0.0,35036.000000,35036.000000,35036.000000,35036.000000,24542.000000
mean,5.007097e+17,2.026041e+13,NaN,1.855502e+08,1.467166e+18,NaN,8.454264,5.587613,6.939554,5.417710,...,4.834390,4.818834,4.745455,4.63393,NaN,54.826008,46.960212,7.034935,0.073467,0.778935
std,5.815799e+17,3.906306e-03,NaN,2.041843e+08,1.219032e+16,NaN,3.719542,3.379174,3.679800,3.299007,...,0.382086,0.424163,0.396114,0.50263,NaN,194.285084,193.302536,30.893477,0.906802,1.818459
min,6.848000e+03,2.026041e+13,NaN,1.678000e+03,1.462506e+18,NaN,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,NaN,1.000000,0.000000,0.000000,0.000000,0.010000
25%,2.158128e+07,2.026041e+13,NaN,1.965853e+07,1.462878e+18,NaN,6.000000,3.000000,4.000000,3.000000,...,4.820000,4.820000,4.660000,4.52000,NaN,1.000000,0.000000,0.000000,0.000000,0.070000
50%,5.148317e+07,2.026041e+13,NaN,9.954958e+07,1.465524e+18,NaN,9.000000,6.000000,7.000000,5.000000,...,4.950000,4.960000,4.860000,4.76000,NaN,1.000000,1.000000,0.000000,0.000000,0.230000
75%,1.030399e+18,2.026041e+13,NaN,3.378483e+08,1.469275e+18,NaN,11.000000,9.000000,10.000000,8.000000,...,5.000000,5.000000,5.000000,4.94000,NaN,8.000000,2.000000,2.000000,0.000000,0.830000
max,1.654399e+18,2.026041e+13,NaN,7.545790e+08,1.649332e+18,NaN,17.000000,11.000000,15.000000,11.000000,...,5.000000,5.000000,5.000000,5.00000,NaN,1090.000000,1090.000000,230.000000,20.000000,116.800000


In [7]:
listing_data.duplicated().sum()

np.int64(0)

In [8]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

                                              missing_percent
id                                                   0.000000
listing_url                                          0.000000
scrape_id                                            0.000000
last_scraped                                         0.000000
source                                               0.000000
name                                                 0.005708
description                                          4.335541
neighborhood_overview                              100.000000
picture_url                                          0.385318
host_id                                              0.000000
host_url                                             0.000000
host_profile_id                                      2.397534
host_profile_url                                     2.580203
host_name                                            2.383263
host_since                                         100.000000
hosts_ti

In [9]:
# Summary of the listing dataset

# - The DataFrame has 36445 rows and 85 columns
# - The data dont have any duplicate value, but it has Nan values.

# - Change datatypes = {
#                       has_availability (str <-> Boolean)
#                       host_has_profile_pic (str <-> Boolean)
#                       host_identity_verified (str <-> Boolean)
#                       host_is_superhost (str <-> Boolean)
#
#                       first_review (str <-> datetime)
#                       bathrooms_text (str <-> float)
#                       amenities (list <-> Json-like string)
#                       }

# - As we have to prepare the data to train a model that predicts Nightly price of a property
#   Therefore, we have to drop few columns

In [10]:
#Dropping columns
cols_to_drop = [
    # 100% null
    'neighborhood_overview', 'host_since', 'host_response_time',
    'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'neighbourhood', 'calendar_updated', 'estimated_revenue_l365d',
    'instant_bookable',
    # URLs and IDs
    'listing_url', 'scrape_id', 'last_scraped', 'picture_url',
    'host_url', 'host_profile_id', 'host_profile_url',
    'host_picture_url', 'calendar_last_scraped',
    # free text
    'name', 'description', 'host_about', 'host_name',
    # high null + not recoverable
    'license', 'host_location',
]

listing_data = listing_data.drop(columns=cols_to_drop)

listing_data = listing_data.drop(columns=['beds'])

listing_data = listing_data.drop(columns=[
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights'
])

listing_data = listing_data.drop(columns=['source'])
print(listing_data.shape)


(35036, 56)


In [11]:
listing_data['bathrooms_text'].value_counts().head(20)

bathrooms_text
1 bath               17648
1 shared bath         7045
1 private bath        4133
2 baths               2435
1.5 baths              937
2 shared baths         860
1.5 shared baths       536
2.5 baths              319
3 baths                246
0 shared baths         223
0 baths                 99
3.5 baths               86
4 baths                 54
2.5 shared baths        52
Half-bath               48
3 shared baths          48
Private half-bath       28
Shared half-bath        27
4 shared baths          27
4.5 baths               21
Name: count, dtype: int64

In [12]:
#Changing Datatypes
bool_cols = ['has_availability', 'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']

for col in bool_cols:
    listing_data[col] = listing_data[col].str.strip().str.lower().map({'t': True, 'f': False})

listing_data['first_review'] = pd.to_datetime(listing_data['first_review'])
listing_data['last_review'] = pd.to_datetime(listing_data['last_review'])

#converting bathroom_text to float values
def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text).lower().strip()
    
    # half bath = 0.5
    if 'half' in text:
        return 0.5
    
    # extract the first number (handles 1, 1.5, 2 etc.)
    match = re.search(r'[\d.]+', text)
    if match:
        return float(match.group())
    
    return np.nan

listing_data['bathrooms_parsed'] = listing_data['bathrooms_text'].apply(parse_bathrooms)
listing_data = listing_data.drop(columns=['bathrooms'])
listing_data = listing_data.rename(columns={'bathrooms_parsed': 'bathrooms'})
listing_data = listing_data.drop(columns=['bathrooms_text'])
#placing bathroom_prased before bedrooms
cols = list(listing_data.columns)
cols.remove('bathrooms')
bedrooms_idx = cols.index('bedrooms')
cols.insert(bedrooms_idx, 'bathrooms')
listing_data = listing_data[cols]


In [13]:
listing_data = listing_data.drop(columns=['host_has_profile_pic', 'host_identity_verified'])

In [14]:
listing_data = listing_data.drop(columns=[
    'price_quote_checkin_date',
    'price_quote_checkout_date', 
    'price_quote_total_price',
    'price_quote_price_per_night',
    'price_quote_raw'
])

In [15]:
listing_data['price'] = listing_data['price'].str.replace('[$,]', '', regex=True).astype(float)

In [16]:
listing_data['price'].describe()

count    20693.000000
mean       254.998857
std        440.478328
min          4.470000
25%         93.800000
50%        167.250000
75%        279.500000
max      15075.000000
Name: price, dtype: float64

In [17]:
print(listing_data.head(10).to_string())

      id  host_id  hosts_time_as_user_years  hosts_time_as_user_months  hosts_time_as_host_years  hosts_time_as_host_months host_is_superhost  host_listings_count neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [18]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

                                              missing_percent
id                                                   0.000000
host_id                                              0.000000
hosts_time_as_user_years                             2.397534
hosts_time_as_user_months                            2.397534
hosts_time_as_host_years                             2.397534
hosts_time_as_host_months                            2.397534
host_is_superhost                                    2.397534
host_listings_count                                  2.397534
neighbourhood_cleansed                               0.000000
neighbourhood_group_cleansed                         0.000000
latitude                                             0.000000
longitude                                            0.000000
property_type                                        0.000000
room_type                                            0.000000
accommodates                                         0.000000
bathroom

In [19]:
# after seeing the null % summary again, we can see that only a few columns has significant null %
# therefore, for price - we have to drop the rows, as 40% is very high
#                bedrooms - fill it with median grouped by property_type
#                bathrooms -  fill it with median grouped by room_type
#                for all the host_time_as_user/host_years/month - fill it with median ()
#                for all the host_is_superhost - fill it with mode
#                has_availability - fill it with mode
#                first and last review - we can modify it to days_since_reviewed and fill the nan values with a large number
#                review scores_* -can't fill it with zero (as it means terrible listing), so will create has_reviews flag to preserve previous value and then fill null values with median
#                reviews_per_month - fill it with zero

In [20]:
#filling null values

listing_data = listing_data.dropna(subset=['price'])

host_numeric_column = ['hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_listings_count']
for col in host_numeric_column:
    listing_data[col] = listing_data[col].fillna(listing_data[col].median())

listing_data['host_is_superhost'] = listing_data['host_is_superhost'].fillna(listing_data['host_is_superhost'].mode()[0])

listing_data['bathrooms'] = listing_data.groupby('room_type')['bathrooms'].transform(lambda x: x.fillna(x.median()))

listing_data['bedrooms'] = listing_data.groupby('property_type')['bedrooms'].transform(lambda x: x.fillna(x.median()))

listing_data['has_availability'] = listing_data['has_availability'].fillna(listing_data['has_availability'].mode()[0])

listing_data['reviews_per_month'] = listing_data['reviews_per_month'].fillna(0)

listing_data['has_reviews'] = listing_data['review_scores_rating'].notna().astype(int)
cols = list(listing_data.columns)
cols.remove('has_reviews')
rating_idx = cols.index('review_scores_rating')
cols.insert(rating_idx, 'has_reviews')
listing_data = listing_data[cols]

review_cols = ['review_scores_rating', 'review_scores_accuracy', 
               'review_scores_cleanliness', 'review_scores_checkin',
               'review_scores_communication', 'review_scores_location', 
               'review_scores_value']

for col in review_cols:
    listing_data[col] = listing_data[col].fillna(listing_data[col].median())

today = pd.Timestamp.today()

listing_data['days_since_first_review'] = (today - listing_data['first_review']).dt.days
listing_data['days_since_last_review'] = (today - listing_data['last_review']).dt.days

listing_data['days_since_first_review'] = listing_data['days_since_first_review'].fillna(99999)
listing_data['days_since_last_review'] = listing_data['days_since_last_review'].fillna(99999)

cols = list(listing_data.columns)
cols.remove('days_since_first_review')
cols.remove('days_since_last_review')
first_idx = cols.index('first_review')
cols.insert(first_idx, 'days_since_first_review')
cols.insert(first_idx + 1, 'days_since_last_review')
listing_data = listing_data[cols]

listing_data = listing_data.drop(columns=['first_review', 'last_review'])



In [21]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

                                              missing_percent
id                                                   0.000000
host_id                                              0.000000
hosts_time_as_user_years                             0.000000
hosts_time_as_user_months                            0.000000
hosts_time_as_host_years                             0.000000
hosts_time_as_host_months                            0.000000
host_is_superhost                                    0.000000
host_listings_count                                  0.000000
neighbourhood_cleansed                               0.000000
neighbourhood_group_cleansed                         0.000000
latitude                                             0.000000
longitude                                            0.000000
property_type                                        0.000000
room_type                                            0.000000
accommodates                                         0.000000
bathroom

In [22]:
#removing tiny remaining nulls from few columns 
listing_data = listing_data.dropna(subset=['bedrooms', 'minimum_nights', 'has_availability'])

In [23]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

                                              missing_percent
id                                                        0.0
host_id                                                   0.0
hosts_time_as_user_years                                  0.0
hosts_time_as_user_months                                 0.0
hosts_time_as_host_years                                  0.0
hosts_time_as_host_months                                 0.0
host_is_superhost                                         0.0
host_listings_count                                       0.0
neighbourhood_cleansed                                    0.0
neighbourhood_group_cleansed                              0.0
latitude                                                  0.0
longitude                                                 0.0
property_type                                             0.0
room_type                                                 0.0
accommodates                                              0.0
bathroom

In [24]:
listing_data.shape

(20680, 49)

In [25]:
#modifying the amenities 
print(type(listing_data['amenities'][0]))
print(listing_data['amenities'][0])

<class 'str'>
["Microwave", "Wifi", "Hangers", "Essentials", "Stove", "Dishwasher", "Free street parking", "Dishes and silverware", "Smoke alarm", "Bed linens", "Iron", "Carbon monoxide alarm", "Heating", "Shampoo", "Kitchen", "TV", "Fire extinguisher", "Oven", "Hot water", "Cooking basics", "Coffee maker", "Refrigerator", "Washer", "Hair dryer", "Air conditioning", "Extra pillows and blankets"]


In [26]:
#prasing the whole column to actual lists
listing_data['amenities'] = listing_data['amenities'].apply(json.loads)

listing_data['amenity_count'] = listing_data['amenities'].apply(len)

In [27]:
listing_data['amenities'].explode().value_counts().head(20)

amenities
Smoke alarm              19555
Wifi                     19181
Carbon monoxide alarm    17713
Kitchen                  17065
Hot water                16004
Hangers                  15254
Essentials               14736
Hair dryer               14170
Iron                     14145
Air conditioning         14063
Bed linens               13969
Dishes and silverware    13397
Refrigerator             12887
Dedicated workspace      12611
Microwave                12387
Heating                  12331
Cooking basics           12300
TV                       11598
Shampoo                  10938
Fire extinguisher        10852
Name: count, dtype: int64

In [28]:
amen_counts = listing_data['amenities'].explode().value_counts()
wanted = ['Pool', 'Gym', 'Elevator', 'Parking', 'Washer', 'Dishwasher']
amen_counts.reindex(wanted, fill_value=0).sort_values(ascending=False)

amenities
Washer        5420
Dishwasher    5372
Elevator      4469
Gym           1865
Pool           249
Parking          0
Name: count, dtype: int64

In [29]:
top_amenities = ['Wifi', 'Kitchen', 'Air conditioning', 
                 'Dedicated workspace', 'TV', 'Heating', 
                 'Refrigerator', 'Microwave', 'Washer', 
                 'Dishwasher', 'Elevator', 'Gym']

for amenity in top_amenities:
    col_name = f"has_{amenity.lower().replace(' ', '_')}"
    listing_data[col_name] = listing_data['amenities'].apply(lambda x: 1 if amenity in x else 0)

In [30]:
has_cols = [c for c in listing_data.columns if c.startswith('has_')]
print(has_cols)
listing_data[has_cols].head()

['has_availability', 'has_reviews', 'has_wifi', 'has_kitchen', 'has_air_conditioning', 'has_dedicated_workspace', 'has_tv', 'has_heating', 'has_refrigerator', 'has_microwave', 'has_washer', 'has_dishwasher', 'has_elevator', 'has_gym']


,has_availability,has_reviews,has_wifi,has_kitchen,has_air_conditioning,has_dedicated_workspace,has_tv,has_heating,has_refrigerator,has_microwave,has_washer,has_dishwasher,has_elevator,has_gym
0,True,1,1,1,1,0,1,1,1,1,1,1,0,0
1,True,1,1,1,0,0,0,1,0,0,1,0,1,0
2,True,1,1,1,1,0,1,1,1,0,0,0,0,0
3,True,1,1,1,0,1,0,1,1,1,0,0,0,0
4,True,1,0,1,0,1,0,1,1,1,0,1,0,0


In [31]:
listing_data = listing_data.drop(columns=['amenities'])

In [32]:
#saving the cleaned file

listing_data.to_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\data\processed\listing_clean.csv', index=False)
print(listing_data.shape)

(20680, 61)
